# Cloud Logging Incident Analysis

Goal: analyze service logs to identify error spikes, high latency, and likely incident sources.

In [ ]:
from pathlib import Path
import pandas as pd

logs = pd.read_csv(Path('../data/cloud_logs_sample.csv'), parse_dates=['timestamp'])
logs['is_error'] = logs['status_code'] >= 500
logs['is_slow'] = logs['latency_ms'] >= 500
logs

In [ ]:
service_summary = logs.groupby('service', as_index=False).agg(
    requests=('status_code', 'count'),
    errors=('is_error', 'sum'),
    slow_requests=('is_slow', 'sum'),
    p95_latency_ms=('latency_ms', lambda s: s.quantile(0.95))
)
service_summary['error_rate'] = service_summary['errors'] / service_summary['requests']
service_summary.sort_values(['errors', 'p95_latency_ms'], ascending=False)

In [ ]:
incident_candidates = logs[(logs['is_error']) | (logs['is_slow'])]
incident_candidates[['timestamp', 'service', 'severity', 'status_code', 'latency_ms', 'message']]

## Interview Talking Points

- Start with service-level error rate and p95 latency.
- Correlate severity, status code, and message patterns.
- Convert findings into alerts and dashboards.